# EventHub AI Create Event V2 — Pipeline 02: Prepare SFT Dataset
### Mục tiêu:
1. Đọc dữ liệu từ 3 split đã được kiểm tra tính độc lập (Train / Validation / Test).
2. Chuyển đổi toàn bộ các mẫu sang định dạng SFT tiêu chuẩn của Hugging Face / TRL (`messages`: system, user, assistant).
3. Sử dụng đúng Chat Template chính thức của Qwen3 (`<|im_start|>` / `<|im_end|>`).
4. Phân tích phân phối độ dài Token (Token Length Distribution) và kiểm tra nguy cơ bị cắt cụt (truncation).
5. Lưu tệp SFT sẵn sàng cho quá trình huấn luyện QLoRA.



In [ ]:
# [1] Cài đặt thư viện và Mount Drive
!pip install -q transformers tiktoken

from google.colab import drive
import os
import json
import matplotlib.pyplot as plt
from transformers import AutoTokenizer

drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/EventHub_AI'
SPLITS_DIR = os.path.join(DRIVE_ROOT, 'dataset/cleaned/eventhub_v2/splits')
SFT_OUT_DIR = os.path.join(DRIVE_ROOT, 'dataset/cleaned/eventhub_v2/sft')
os.makedirs(SFT_OUT_DIR, exist_ok=True)



In [ ]:
# [2] Thiết lập System Instruction chuẩn hóa chống Hallucination
SYSTEM_INSTRUCTION = """Bạn là chuyên gia trích xuất thông tin sự kiện của hệ thống EventHub.
Nhiệm vụ của bạn là đọc kỹ văn bản tài liệu sự kiện được cung cấp và trích xuất thông tin theo đúng định dạng JSON tuân thủ schema quy định.

Quy tắc bắt buộc:
1. Chỉ trích xuất thông tin có căn cứ trực tiếp trong tài liệu. Tuyệt đối không bịa đặt hoặc suy diễn thông tin (không hallucination).
2. Các trường không có thông tin hoặc chưa xác định phải gán giá trị null (hoặc mảng rỗng [] đối với tags).
3. Định dạng ngày tháng: YYYY-MM-DD. Định dạng giờ: HH:mm (24 giờ).
4. Phân biệt rõ giờ check-in (checkin_start_time) và giờ bắt đầu sự kiện (start_time).
5. Nếu sự kiện diễn ra nhiều ngày hoặc nhiều phiên, trích xuất đầy đủ từng session trong mảng "sessions".
6. Toàn bộ các hạng vé, giá vé (VNĐ, số nguyên) và số lượng phải được đưa vào mảng "ticket_types". Vé miễn phí ghi price là 0.
7. Chỉ trả về duy nhất chuỗi JSON hợp lệ, bắt đầu bằng { và kết thúc bằng }. Tuyệt đối không thêm lời giải thích hoặc markdown code block bên ngoài.""".strip()



In [ ]:
# [3] Tải Tokenizer chính thức của Qwen
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct" # Dùng tokenizer tương thích của dòng Qwen
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
print(f"Đã tải thành công Tokenizer từ: {MODEL_ID}")
print(f"Chat template khả dụng: {tokenizer.chat_template is not None}")



In [ ]:
# [4] Chuyển đổi và đo lường độ dài Token của từng Split
all_token_counts = []

for split_name in ["train", "validation", "test"]:
    in_file = os.path.join(SPLITS_DIR, f"{split_name}.jsonl")
    out_file = os.path.join(SFT_OUT_DIR, f"{split_name}_sft.jsonl")

    if not os.path.exists(in_file):
        print(f"Bỏ qua {in_file} (không tìm thấy file)")
        continue

    split_tokens = []
    converted = []

    with open(in_file, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rec = json.loads(line)
                user_msg = rec.get("input_text", "").strip()
                out_obj = rec.get("output_json", {})
                asst_msg = json.dumps(out_obj, ensure_ascii=False, indent=2)

                messages = [
                    {"role": "system", "content": SYSTEM_INSTRUCTION},
                    {"role": "user", "content": user_msg},
                    {"role": "assistant", "content": asst_msg}
                ]

                # Áp dụng chat template để tính chính xác token huấn luyện
                formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False)
                tokens = tokenizer.encode(formatted_prompt)
                token_len = len(tokens)

                split_tokens.append(token_len)
                all_token_counts.append(token_len)

                converted.append({
                    "id": rec.get("id"),
                    "split": split_name,
                    "type": rec.get("type"),
                    "messages": messages
                })

    with open(out_file, 'w', encoding='utf-8') as f:
        for c in converted:
            f.write(json.dumps(c, ensure_ascii=False) + '\n')

    print(f"[{split_name.upper()}] Đã chuyển đổi {len(converted)} mẫu -> {out_file}")
    print(f"  - Min tokens: {min(split_tokens)} | Max tokens: {max(split_tokens)} | Avg tokens: {sum(split_tokens)/len(split_tokens):.1f}")



In [ ]:
# [5] Vẽ biểu đồ phân phối độ dài Token & Kiểm tra Truncation (ngưỡng >= 4096)
if all_token_counts:
    plt.figure(figsize=(10, 5))
    plt.hist(all_token_counts, bins=25, color='royalblue', edgecolor='black', alpha=0.7)
    plt.axvline(4096, color='red', linestyle='--', linewidth=2, label='Ngưỡng Context 4096 tokens')
    plt.axvline(6144, color='orange', linestyle=':', linewidth=2, label='Ngưỡng Colab L4/T4 6144 tokens')
    plt.title('Phân phối độ dài Token (Chat Template Qwen) trên toàn bộ Dataset')
    plt.xlabel('Số lượng tokens')
    plt.ylabel('Số lượng mẫu')
    plt.legend()
    plt.grid(axis='y', alpha=0.5)
    plt.show()

    max_tok = max(all_token_counts)
    print(f"Token dài nhất: {max_tok}")
    if max_tok <= 4096:
        print("✅ AN TOÀN TUYỆT ĐỐI: 100% mẫu huấn luyện <= 4096 tokens, vừa khít RTX 3070 8GB & Colab T4/L4.")
    elif max_tok <= 6144:
        print(f"⚠️ LƯU Ý: Có mẫu từ 4097..6144 tokens ({max_tok}). Dùng max_seq_length = 6144 trên Colab L4/T4.")
    else:
        print(f"❌ CẢNH BÁO: Có mẫu vượt quá 6144 tokens ({max_tok}). Cần rút gọn mô tả HTML trước khi train.")

